## 3 Errors (Insertion, Deletion, and Replacement) + multiple subsets

### Print 50 CTB samples before noise application

In [21]:
def preview_ctb_samples(seg_dir, id_min, id_max, n_samples=50):
    """
    Preview n_samples sentences from the CTB subset before noise is applied.
    Useful to inspect structure and segmentation quality.
    """
    sents = load_ctb_segmented(seg_dir, id_min=id_min, id_max=id_max)
    print(f"\nTotal usable sentences in CTB ID range [{id_min}, {id_max}]: {len(sents)}")

    # If n_samples > total, adjust
    n_samples = min(n_samples, len(sents))

    print(f"\nShowing {n_samples} random samples:\n")

    sample_indices = random.sample(range(len(sents)), n_samples)

    for idx in sample_indices:
        ctb_id, tokens = sents[idx]
        print(f"[{ctb_id}]  {' '.join(tokens)}")
        print("-" * 80)


In [23]:
if __name__ == "__main__":
    random.seed(42)

    # Preview 20 sentences from CTB range 400–550
    preview_ctb_samples(CTB_SEG_DIR, CTB_ID_MIN, CTB_ID_MAX, n_samples=50)

    # Then continue with actual synthetic data creation
    df_ctb = build_synthetic_ctb_df(
        CTB_SEG_DIR,
        SYNTH_SIZE,
        NOISE_RATIO,
        id_min=CTB_ID_MIN,
        id_max=CTB_ID_MAX,
    )



Total usable sentences in CTB ID range [400, 550]: 1390

Showing 50 random samples:

[chtb_544.seg:63]  入境处 人员 会 根据 由 内地 公安局 提供 的 资料 及 对 儿童 出生 时 其 父母 在 港 的 居留 身分 的 纪录 加以 审查 ， 以 评估 该 名 人士 是否 根据 《 基本法 》 第二十四 条 第二 款 第三 项 而 享有 居留权 。
--------------------------------------------------------------------------------
[chtb_417.seg:21]  同时 ， 包装 行业 获 世界 之 星 “ 包装 设计 ” 奖 八十余 项 。
--------------------------------------------------------------------------------
[chtb_403.seg:19]  中国 人民 银行 外资 金融 机构 管理司 副司长 张循海 在 今天 举行 的 “ 中国 和 拉美 金融 体制 改革 国际 研讨会 ” 上 说 ， 一九七九年 第一 家 外国 银行 代表处 在 中国 设立 以来 ， 进入 中国 的 外资 金融 机构 不仅 在 数量 上 ， 而且 在 业务 规模 上 都 已 成为 中国 金融 体系 中 重要 组成 部分 。
--------------------------------------------------------------------------------
[chtb_440.seg:21]  成交 的 主要 商品 是 草柳 藤竹 制品 ， 还有 电子 产品 、 陶瓷 、 玩具 、 服装 、 鞋 帽 、 箱 包 、 家具 等 。
--------------------------------------------------------------------------------
[chtb_436.seg:15]  记者 从 正 在 天津 经济 技术 开发区 举行 的 中国 银行 对 开发区 及 外商 投资 企业 信贷 管理 与 发展 座谈会 上 获悉 ： 中国 银行 正在 采

### Synthetic Data Generator with error distribution from learner's corpus

In [27]:
import os
import glob
import random
import re
import pandas as pd
from typing import List, Tuple

# =========================================
# 1. CONFIG
# =========================================

# Path to the CTB 5.1 "segmented" directory (EDIT THIS)
CTB_SEG_DIR = "ctb5.1_507K/segmented"

# Number of sentences to sample from CTB subset
SYNTH_SIZE = 1079   # match learner corpus size

# Use only files whose numeric ID is within this range:
# e.g. chtb_0400.seg ... chtb_0550.seg
CTB_ID_MIN = 400
CTB_ID_MAX = 550

# Error type distribution from learner corpus
# substitution -> REPLACE, deletion -> DELETE, insertion -> INSERT
ERROR_DIST = {
    "delete": 0.318,
    "insert": 0.199,
    "replace": 0.483,
}

# Sanity: sum should be 1.0 (or very close)
assert abs(sum(ERROR_DIST.values()) - 1.0) < 1e-6, "ERROR_DIST must sum to 1.0"


# =========================================
# 2. BASIC HELPERS
# =========================================

def has_cjk(s: str) -> bool:
    """Return True if the string contains at least one CJK character."""
    for ch in s:
        if "\u4e00" <= ch <= "\u9fff":
            return True
    return False


# =========================================
# 3. LOAD CTB SEGMENTED SENTENCES (MULTI-FILE SUBSET)
# =========================================

def load_ctb_segmented(seg_dir: str,
                       id_min: int = None,
                       id_max: int = None) -> List[Tuple[str, List[str]]]:
    """
    Read *.seg files in CTB segmented directory, restricted to a numeric
    ID range if given (e.g. 400–450 => chtb_0400.seg ... chtb_0450.seg).

    Each non-empty line that looks like a real sentence is treated as one
    segmented sentence.

    We SKIP:
      - pure markup lines like <DOCID>...</DOCID>, <S>, </S>, etc.
      - lines with no Chinese characters
      - CTB meta markers like "完" / "（ 完 ）"
      - Xinhua dateline/header lines starting with "新华社 "
    """
    sent_list: List[Tuple[str, List[str]]] = []

    pattern = os.path.join(seg_dir, "chtb_*.seg")
    all_seg_files = sorted(glob.glob(pattern))

    if not all_seg_files:
        raise FileNotFoundError(f"No .seg files found under {seg_dir}")

    # filter by numeric ID range
    seg_files = []
    for seg_path in all_seg_files:
        fname = os.path.basename(seg_path)  # e.g. "chtb_0400.seg"
        m = re.search(r"(\d+)", fname)
        if m is None:
            continue
        num_id = int(m.group(1))
        if id_min is not None and num_id < id_min:
            continue
        if id_max is not None and num_id > id_max:
            continue
        seg_files.append(seg_path)

    if not seg_files:
        raise FileNotFoundError(
            f"No .seg files in {seg_dir} within ID range [{id_min}, {id_max}]"
        )

    # known CTB meta marker patterns we want to drop
    META_LINES = {
        "完",
        "（ 完 ）",   # full-width parens
        "( 完 )",     # ascii parens variant (defensive)
        "（ 完)",     # mixed paren variants
        "(完 ）",
    }

    for seg_path in seg_files:
        fname = os.path.basename(seg_path)

        # CTB5.1 is GB-encoded; ignore invalid bytes
        with open(seg_path, "r", encoding="gb18030", errors="ignore") as f:
            for line_no, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue

                # Skip markup lines like <DOCID>...</DOCID>, <S>, </S>, etc.
                if line.startswith("<") and line.endswith(">"):
                    continue
                tokens = line.split()
                if not tokens:
                    continue
                if all(tok.startswith("<") and tok.endswith(">") for tok in tokens):
                    continue

                # Skip lines with no Chinese characters
                if not has_cjk(line):
                    continue

                # Skip CTB meta markers ("完", "（ 完 ）", etc.)
                if line in META_LINES:
                    continue

                # Skip Xinhua dateline/header sentences
                # (e.g. "新华社 北京 十月 十三日 电 （ 记者 ..." )
                if line.startswith("新华社 "):
                    continue

                # Now treat as a real sentence
                tokens = line.split()
                ctb_id = f"{fname}:{line_no}"
                sent_list.append((ctb_id, tokens))

    return sent_list


# =========================================
# 4. GLOBAL CHARACTER-LEVEL NOISE
#    with learner-style distribution
#    (INSERT / DELETE / REPLACE)
# =========================================

def _sample_op() -> str:
    """Sample an operation name according to ERROR_DIST."""
    r = random.random()
    p_del = ERROR_DIST["delete"]
    p_ins = ERROR_DIST["insert"]
    # p_rep = ERROR_DIST["replace"]  # implied

    if r < p_del:
        return "delete"
    elif r < p_del + p_ins:
        return "insert"
    else:
        return "replace"


def apply_global_char_noise(
    all_tokens: List[List[str]],
    noise_ratio: float
) -> List[List[str]]:
    """
    Apply character-level noise globally so that the total number of
    edit operations ~= noise_ratio * total_characters.

    Noise operations (with learner-style distribution):
      - DELETE: proportion ~ 0.318
      - INSERT: proportion ~ 0.199
      - REPLACE: proportion ~ 0.483

    Noise is DEFINED AT CHARACTER LEVEL:
      total_chars = sum(len(tok) for all tokens before noise)
      num_ops = round(noise_ratio * total_chars)
      Each op touches exactly ONE character position.

    Returns:
        New list of sentences (same structure) with noisy tokens.
    """

    # tokens -> chars
    token_chars = [[list(tok) for tok in sent] for sent in all_tokens]

    total_chars = sum(len(tok) for sent in all_tokens for tok in sent)
    if total_chars == 0:
        return all_tokens

    num_ops = max(1, int(round(noise_ratio * total_chars)))

    # pool of chars for insertion/replacement
    char_pool = [ch for sent in all_tokens for tok in sent for ch in tok if ch.strip()]
    if not char_pool:
        return all_tokens

    num_sents = len(token_chars)
    if num_sents == 0:
        return all_tokens

    for _ in range(num_ops):
        si = random.randrange(num_sents)
        sent = token_chars[si]
        if not sent:
            continue

        # Sample primary op using learner error distribution
        primary = _sample_op()
        # Define a fallback order (if primary not applicable)
        if primary == "delete":
            ops_order = ["delete", "replace", "insert"]
        elif primary == "insert":
            ops_order = ["insert", "replace", "delete"]
        else:  # "replace"
            ops_order = ["replace", "delete", "insert"]

        applied = False
        for op_type in ops_order:
            # DELETE
            if op_type == "delete":
                del_candidates = [
                    (ti, len(chars)) for ti, chars in enumerate(sent) if len(chars) >= 1
                ]
                if del_candidates:
                    ti, length = random.choice(del_candidates)
                    ci = random.randrange(length)
                    del sent[ti][ci]
                    applied = True
                    break

            # REPLACE
            elif op_type == "replace":
                rep_candidates = [
                    (ti, len(chars)) for ti, chars in enumerate(sent) if len(chars) >= 1
                ]
                if rep_candidates:
                    ti, length = random.choice(rep_candidates)
                    ci = random.randrange(length)
                    old_ch = sent[ti][ci]
                    new_ch = random.choice(char_pool)
                    attempts = 0
                    while new_ch == old_ch and attempts < 3:
                        new_ch = random.choice(char_pool)
                        attempts += 1
                    sent[ti][ci] = new_ch
                    applied = True
                    break

            # INSERT
            elif op_type == "insert":
                ins_candidates = [
                    (ti, len(chars)) for ti, chars in enumerate(sent) if len(chars) >= 0
                ]
                if ins_candidates:
                    ti, length = random.choice(ins_candidates)
                    pos = random.randrange(length + 1)
                    ch = random.choice(char_pool)
                    sent[ti].insert(pos, ch)
                    applied = True
                    break

        if not applied:
            # No valid operation possible for this sentence in this iteration
            continue

    noisy_all_tokens: List[List[str]] = [
        ["".join(chars) for chars in sent] for sent in token_chars
    ]
    return noisy_all_tokens


# =========================================
# 5. MAIN: generate 10 datasets (1%–10%)
# =========================================

if __name__ == "__main__":
    # 1) Load and fix the CTB sentence subset ONCE (no noise yet)
    print(f"Loading CTB sentences from IDs [{CTB_ID_MIN}, {CTB_ID_MAX}] ...")
    all_sents = load_ctb_segmented(CTB_SEG_DIR, id_min=CTB_ID_MIN, id_max=CTB_ID_MAX)
    n_total = len(all_sents)
    print(f"Loaded {n_total} usable sentences.")

    if SYNTH_SIZE > n_total:
        raise ValueError(
            f"Requested {SYNTH_SIZE} sentences, but only {n_total} available "
            f"in CTB IDs [{CTB_ID_MIN}, {CTB_ID_MAX}] after filtering. "
            f"Please widen CTB_ID_MIN/CTB_ID_MAX."
        )

    # 2) Fix the base sample of 1079 sentences (same for all noise levels)
    BASE_SAMPLE_SEED = 42
    random.seed(BASE_SAMPLE_SEED)
    sampled_indices = random.sample(range(n_total), SYNTH_SIZE)
    sampled_ids = [all_sents[i][0] for i in sampled_indices]
    sampled_tokens = [all_sents[i][1] for i in sampled_indices]

    print(f"Sampled {SYNTH_SIZE} base sentences (fixed for all noise levels).")

    # 3) Generate 10 datasets with 1%–10% noise
    for noise_pct in range(1, 11):  # 1..10
        noise_ratio = noise_pct / 100.0

        # Different fixed seed per noise level for training diversity
        random.seed(noise_pct)

        print(f"\n=== Generating dataset with {noise_pct}% noise (ratio={noise_ratio}) ===")

        # Apply learner-style char-level noise
        noisy_tokens = apply_global_char_noise(sampled_tokens, noise_ratio)

        # Build rows
        rows = []
        for idx, (ctb_id, clean_tok, noisy_tok) in enumerate(
            zip(sampled_ids, sampled_tokens, noisy_tokens)
        ):
            target_str = " ".join(clean_tok)    # original CTB segmentation
            gold_str = " ".join(noisy_tok)      # noisy source

            rows.append(
                {
                    "Index": idx,
                    "CTB_ID": ctb_id,
                    "Gold_Source": gold_str,   # noisy
                    "Target_Text": target_str  # clean
                }
            )

        df = pd.DataFrame(rows)

        output_path = (
            f"ctb5_synthetic_charnoise_{noise_pct}pct_ids{CTB_ID_MIN}_{CTB_ID_MAX}_{SYNTH_SIZE}.csv"
        )
        df.to_csv(output_path, index=False, encoding="utf-8-sig")

        print(
            f"Saved synthetic CTB learner corpus "
            f"({len(df)} sentences, {noise_pct}% char-level noise) "
            f"to {output_path}"
        )
        print(df.head())


Loading CTB sentences from IDs [400, 550] ...
Loaded 1230 usable sentences.
Sampled 1079 base sentences (fixed for all noise levels).

=== Generating dataset with 1% noise (ratio=0.01) ===
Saved synthetic CTB learner corpus (1079 sentences, 1% char-level noise) to ctb5_synthetic_charnoise_1pct_ids400_550_1079.csv
   Index           CTB_ID                                        Gold_Source  \
0      0  chtb_421.seg:15  记者 从 中国 国家 经济 贸易 委员会 日前 在 兰州 召开 的 全 国 新 技术 推广 ...   
1      1  chtb_403.seg:41     中国 目前 正 处于 由 计划 经济 向 社会主义 市场 经济 体制 的 转轨 过程 中 。   
2      2  chtb_446.seg:45  共 接待 了 来自 泰国 、 新加坡 、 印尼 、 马来西亚 、 美国 、 加拿大 、 法国...   
3      3  chtb_442.seg:17  我们 一定 要 举 全 省 之 力 ， 确保 工程 顺利 进行 ， 并 藉 此 促进 湖北 ...   
4      4  chtb_438.seg:25  在 未来 十五 年 内 ， 这 一 断区 的 经济 差不多 会 翻 一 番 ， 世界 经济 ...   

                                         Target_Text  
0  记者 从 中国 国家 经济 贸易 委员会 日前 在 兰州 召开 的 全 国 新 技术 推广 ...  
1     中国 目前 正 处于 由 计划 经济 向 社会主义 市场 经济 体制 的 转轨 过程 中 。  
2  共 接待 了 来自 泰国 、 新加坡 、 印尼 、 马来西亚 、 美国 

### Error Type and Count for Synthetic Data

In [ ]:
import pandas as pd
import difflib
import os

# Same naming pattern as your generator:
CTB_ID_MIN = 400
CTB_ID_MAX = 550
SYNTH_SIZE = 1079

def compute_char_edit_counts(target_str: str, source_str: str):
    """
    Compute character-level edit counts to transform target -> source.

    target_str: clean (Target_Text)
    source_str: noisy (Gold_Source)

    From learner perspective:
      - delete: something present in target but missing in source -> deletion error in source
      - insert: something present in source but not in target -> insertion error in source
      - replace: substitution error in source
    """
    sm = difflib.SequenceMatcher(None, target_str, source_str)

    num_del = 0
    num_ins = 0
    num_sub = 0

    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == "equal":
            continue
        elif tag == "delete":
            # chars removed when going target -> source => missing in source
            num_del += (i2 - i1)
        elif tag == "insert":
            # chars inserted in source
            num_ins += (j2 - j1)
        elif tag == "replace":
            num_sub += max(i2 - i1, j2 - j1)

    return num_sub, num_del, num_ins


for pct in range(1, 11):  # 1% ... 10%
    input_path = f"ctb5_synthetic_charnoise_{pct}pct_ids{CTB_ID_MIN}_{CTB_ID_MAX}_{SYNTH_SIZE}.csv"
    output_path = f"ctb5_synthetic_charnoise_{pct}pct_ids{CTB_ID_MIN}_{CTB_ID_MAX}_{SYNTH_SIZE}_with_errcounts.csv"

    if not os.path.exists(input_path):
        print(f"⚠️ Missing file: {input_path} — skipping")
        continue

    print(f"Processing: {input_path}")

    df = pd.read_csv(input_path, encoding="utf-8-sig")

    df["Gold_Source"] = df["Gold_Source"].fillna("")
    df["Target_Text"] = df["Target_Text"].fillna("")

    num_sub_list = []
    num_del_list = []
    num_ins_list = []
    summary_list = []

    for tgt, src in zip(df["Target_Text"], df["Gold_Source"]):
        sub, dele, ins = compute_char_edit_counts(str(tgt), str(src))
        num_sub_list.append(sub)
        num_del_list.append(dele)
        num_ins_list.append(ins)
        summary_list.append(f"sub={sub},del={dele},ins={ins}")

    # Add new columns
    df["num_sub"] = num_sub_list
    df["num_del"] = num_del_list
    df["num_ins"] = num_ins_list
    # Optional single summary column (what kind + how many)
    df["error_profile"] = summary_list

    df.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"✅ Saved: {output_path}")


Processing: ctb5_synthetic_charnoise_1pct_ids400_550_1079.csv
✅ Saved: ctb5_synthetic_charnoise_1pct_ids400_550_1079_with_errcounts.csv
Processing: ctb5_synthetic_charnoise_2pct_ids400_550_1079.csv
✅ Saved: ctb5_synthetic_charnoise_2pct_ids400_550_1079_with_errcounts.csv
Processing: ctb5_synthetic_charnoise_3pct_ids400_550_1079.csv
✅ Saved: ctb5_synthetic_charnoise_3pct_ids400_550_1079_with_errcounts.csv
Processing: ctb5_synthetic_charnoise_4pct_ids400_550_1079.csv
✅ Saved: ctb5_synthetic_charnoise_4pct_ids400_550_1079_with_errcounts.csv
Processing: ctb5_synthetic_charnoise_5pct_ids400_550_1079.csv
✅ Saved: ctb5_synthetic_charnoise_5pct_ids400_550_1079_with_errcounts.csv
Processing: ctb5_synthetic_charnoise_6pct_ids400_550_1079.csv
✅ Saved: ctb5_synthetic_charnoise_6pct_ids400_550_1079_with_errcounts.csv
Processing: ctb5_synthetic_charnoise_7pct_ids400_550_1079.csv
✅ Saved: ctb5_synthetic_charnoise_7pct_ids400_550_1079_with_errcounts.csv
Processing: ctb5_synthetic_charnoise_8pct_ids400

: 

### Pre-training data generator